# Qwen3-Embedding Fine-Tuning on SPARQL Binary Pairs

This notebook fine-tunes **Qwen/Qwen3-Embedding** separately on:
- `qwen_finetune_dataset/wikidata/` → `checkpoints/qwen_wikidata`
- `qwen_finetune_dataset/dbpedia/`  → `checkpoints/qwen_dbpedia`

### Method
- Loss: **CosineSimilarityLoss** (label=1 → cos→1, label=0 → cos→0)
- Efficient fine-tuning: **LoRA** (rank=16) via PEFT — only ~1% of params trained
- Optional 4-bit quantization via **BitsAndBytes** for GPU memory saving
- Pooling: **last-token** (as used by Qwen3-Embedding natively)
- Evaluation: **BinaryClassificationEvaluator** on val split after each epoch

### Requirements
```
pip install sentence-transformers>=3.0 peft bitsandbytes accelerate transformers datasets
```

In [ ]:
# Run this cell once to install all required packages
import subprocess, sys
pkgs = [
    "sentence-transformers>=3.0",
    "peft",
    "bitsandbytes",
    "accelerate",
    "transformers>=4.40",
    "datasets",
    "torch",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ Packages installed.")

In [ ]:
import os
import json
import torch
import pandas as pd
from pathlib import Path
from torch import nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
    models,
)
from sentence_transformers.evaluation import BinaryClassificationEvaluator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports ready.  Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
BASE_MODEL      = "Qwen/Qwen3-Embedding"   # 0.6B — change to larger if VRAM allows
MAX_SEQ_LENGTH  = 512

# ── LoRA ──────────────────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE      = 16
NUM_EPOCHS      = 3
WARMUP_STEPS    = 100
LR              = 2e-4
SEED            = 42

# ── 4-bit quantization (set False if you have ≥24 GB VRAM) ───────────────
USE_4BIT        = True

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_ROOT       = Path("qwen_finetune_dataset")
CKPT_ROOT       = Path("checkpoints")
CKPT_ROOT.mkdir(exist_ok=True)

DATASETS = {
    "wikidata": {
        "train" : DATA_ROOT / "wikidata" / "train.csv",
        "val"   : DATA_ROOT / "wikidata" / "val.csv",
        "output": CKPT_ROOT / "qwen_wikidata",
    },
    "dbpedia": {
        "train" : DATA_ROOT / "dbpedia" / "train.csv",
        "val"   : DATA_ROOT / "dbpedia" / "val.csv",
        "output": CKPT_ROOT / "qwen_dbpedia",
    },
}

print("✅ Configuration set.")
print(f"   Base model      : {BASE_MODEL}")
print(f"   LoRA rank       : {LORA_R}")
print(f"   Batch size      : {BATCH_SIZE}")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   4-bit quant     : {USE_4BIT}")

In [ ]:
def load_examples(csv_path):
    """Load CSV into SentenceTransformer InputExample list."""
    df = pd.read_csv(csv_path)
    examples = [
        InputExample(
            texts=[row.anchor_query, row.query],
            label=float(row.label),
        )
        for _, row in df.iterrows()
    ]
    print(f"  Loaded {len(examples)} examples from {csv_path.name}  "
          f"(pos={int(df.label.sum())}  neg={int((df.label==0).sum())})")
    return examples

def load_evaluator(csv_path, ds_name):
    """Build a BinaryClassificationEvaluator from val CSV."""
    df = pd.read_csv(csv_path)
    return BinaryClassificationEvaluator(
        sentences1 = df["anchor_query"].tolist(),
        sentences2 = df["query"].tolist(),
        labels     = df["label"].astype(int).tolist(),
        name       = ds_name,
        show_progress_bar = False,
    )

print("✅ Data helpers defined.")

In [ ]:
def build_qwen_sentence_transformer(base_model_name, use_4bit=True, max_seq_length=512):
    """
    Load Qwen3-Embedding with LoRA adapters wrapped as a SentenceTransformer.
    Uses last-token pooling (native to Qwen3-Embedding).
    """
    print(f"Loading tokenizer: {base_model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name, trust_remote_code=True, padding_side="left"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── 4-bit quantization config ─────────────────────────────────────────
    bnb_config = None
    if use_4bit and torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
            bnb_4bit_use_double_quant = True,
        )
        print("  4-bit quantization: ON (nf4 + double quant)")

    print(f"Loading model: {base_model_name}")
    base_model = AutoModel.from_pretrained(
        base_model_name,
        quantization_config = bnb_config,
        torch_dtype         = torch.bfloat16 if not use_4bit else None,
        trust_remote_code   = True,
        device_map          = "auto" if torch.cuda.is_available() else None,
    )

    # ── LoRA config ───────────────────────────────────────────────────────
    lora_config = LoraConfig(
        r                = LORA_R,
        lora_alpha       = LORA_ALPHA,
        lora_dropout     = LORA_DROPOUT,
        target_modules   = LORA_TARGET_MODULES,
        bias             = "none",
        task_type        = TaskType.FEATURE_EXTRACTION,
    )
    model_with_lora = get_peft_model(base_model, lora_config)
    model_with_lora.print_trainable_parameters()

    # ── Wrap as SentenceTransformer ───────────────────────────────────────
    # Custom word embedding module that uses the PEFT model
    word_embedding_model = models.Transformer(
        base_model_name,
        max_seq_length   = max_seq_length,
        tokenizer_args   = {"trust_remote_code": True, "padding_side": "left"},
        model_args       = {"trust_remote_code": True},
    )
    # Replace the underlying model with our LoRA-wrapped version
    word_embedding_model.auto_model = model_with_lora

    # Last-token pooling (required for Qwen3-Embedding)
    pooling_model = models.Pooling(
        word_embedding_model.get_word_embedding_dimension(),
        pooling_mode_lasttoken = True,
        pooling_mode_mean_tokens = False,
    )

    st_model = SentenceTransformer(
        modules=[word_embedding_model, pooling_model],
        device=device,
    )
    return st_model

print("✅ Model builder defined.")

In [ ]:
def train_qwen(ds_name, train_csv, val_csv, output_dir):
    """
    Full training pipeline for one dataset.
    Saves the fine-tuned model to output_dir.
    """
    print("\n" + "="*60)
    print(f"  Fine-tuning Qwen on: {ds_name.upper()}")
    print("="*60)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Load data ─────────────────────────────────────────────────────────
    print("\n[1/4] Loading data...")
    train_examples = load_examples(train_csv)
    evaluator      = load_evaluator(val_csv, ds_name)

    train_dataloader = DataLoader(
        train_examples,
        shuffle    = True,
        batch_size = BATCH_SIZE,
        drop_last  = True,   # avoids issues with very small last batch
    )

    # ── Build model ───────────────────────────────────────────────────────
    print("\n[2/4] Building model with LoRA...")
    model = build_qwen_sentence_transformer(
        BASE_MODEL,
        use_4bit       = USE_4BIT,
        max_seq_length = MAX_SEQ_LENGTH,
    )

    # ── Loss: CosineSimilarityLoss ─────────────────────────────────────────
    # label=1.0 → pushes cosine to 1.0
    # label=0.0 → pushes cosine to 0.0
    train_loss = losses.CosineSimilarityLoss(model)

    # ── Train ─────────────────────────────────────────────────────────────
    print("\n[3/4] Training...")
    total_steps = len(train_dataloader) * NUM_EPOCHS
    print(f"  Steps per epoch : {len(train_dataloader)}")
    print(f"  Total steps     : {total_steps}")
    print(f"  Warmup steps    : {WARMUP_STEPS}")

    model.fit(
        train_objectives     = [(train_dataloader, train_loss)],
        evaluator            = evaluator,
        epochs               = NUM_EPOCHS,
        warmup_steps         = WARMUP_STEPS,
        optimizer_params     = {"lr": LR},
        output_path          = str(output_dir),
        save_best_only       = True,
        evaluation_steps     = len(train_dataloader),  # eval at end of every epoch
        show_progress_bar    = True,
        use_amp              = torch.cuda.is_available(),  # fp16 mixed precision on GPU
        checkpoint_path      = str(output_dir / "checkpoints"),
        checkpoint_save_steps= len(train_dataloader),
    )

    # ── Save final model ──────────────────────────────────────────────────
    print("\n[4/4] Saving final model...")
    model.save(str(output_dir / "final"))
    print(f"  ✅ Model saved → {output_dir / 'final'}")

    # Save training config alongside model
    cfg = {
        "base_model"    : BASE_MODEL,
        "dataset"       : ds_name,
        "lora_r"        : LORA_R,
        "lora_alpha"    : LORA_ALPHA,
        "batch_size"    : BATCH_SIZE,
        "epochs"        : NUM_EPOCHS,
        "lr"            : LR,
        "max_seq_length": MAX_SEQ_LENGTH,
        "use_4bit"      : USE_4BIT,
        "train_pairs"   : len(train_examples),
    }
    with open(output_dir / "training_config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    return model

print("✅ Training function defined.")

In [ ]:
cfg = DATASETS["wikidata"]
model_wikidata = train_qwen(
    ds_name    = "wikidata",
    train_csv  = cfg["train"],
    val_csv    = cfg["val"],
    output_dir = cfg["output"],
)

In [ ]:
# Free Wikidata model from memory before loading DBpedia model
del model_wikidata
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cfg = DATASETS["dbpedia"]
model_dbpedia = train_qwen(
    ds_name    = "dbpedia",
    train_csv  = cfg["train"],
    val_csv    = cfg["val"],
    output_dir = cfg["output"],
)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

results_summary = []

for ds_name, cfg in DATASETS.items():
    print(f"\nEvaluating {ds_name.upper()} model on val set...")
    model = SentenceTransformer(str(cfg["output"] / "final"))
    evaluator = load_evaluator(cfg["val"], ds_name)
    scores = evaluator(model)

    # scores is a dict: metric_name -> value
    print(f"  Results for {ds_name}:")
    for k, v in scores.items():
        print(f"    {k}: {v:.4f}")
    results_summary.append({"dataset": ds_name, **scores})

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_results = pd.DataFrame(results_summary)
print("\n=== Final Evaluation Summary ===")
print(df_results.to_string(index=False))
df_results.to_csv("checkpoints/final_eval_results.csv", index=False)
print("\n✅ Saved → checkpoints/final_eval_results.csv")

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Load one fine-tuned model for inference demo ──────────────────────────
demo_model = SentenceTransformer(str(DATASETS["wikidata"]["output"] / "final"))

# Two semantically equivalent Wikidata queries (should score ~1.0)
q_orig = "SELECT ?x WHERE { ?x wdt:P31 wd:Q5 . ?x wdt:P19 wd:Q60 . }"
q_pert = "SELECT ?v0 WHERE { ?v0 wdt:P19 wd:Q60 . ?v0 wdt:P31 wd:Q5 . }"   # reordered + renamed
q_neg  = "SELECT ?y WHERE { ?y wdt:P31 wd:Q146 . ?y wdt:P17 wd:Q183 . }"   # different query

embs = demo_model.encode([q_orig, q_pert, q_neg], normalize_embeddings=True)

sim_pos = float(np.dot(embs[0], embs[1]))
sim_neg = float(np.dot(embs[0], embs[2]))

print("Inference demo (Wikidata model)")
print("-" * 45)
print(f"Original vs Perturbed (pos): cos = {sim_pos:.4f}  ← should be HIGH")
print(f"Original vs Negative  (neg): cos = {sim_neg:.4f}  ← should be LOW")
print()
print("To use in your evaluation pipeline:")
print("  model_wikidata = SentenceTransformer('checkpoints/qwen_wikidata/final')")
print("  model_dbpedia  = SentenceTransformer('checkpoints/qwen_dbpedia/final')")

## Troubleshooting

| Problem | Fix |
|---------|-----|
| CUDA out of memory | Set `BATCH_SIZE=8` or `USE_4BIT=True` |
| `bitsandbytes` not found | Run `pip install bitsandbytes` and restart kernel |
| Slow on CPU | Add `BATCH_SIZE=4`, `NUM_EPOCHS=1` for a quick smoke test |
| `save_best_only` not recognized | Upgrade: `pip install -U sentence-transformers` |
| Qwen model needs HF token | `huggingface-cli login` before running |

## Memory requirements

| Setup | VRAM needed |
|-------|-------------|
| Qwen3-Emb 0.6B + LoRA + 4-bit | ~6 GB |
| Qwen3-Emb 0.6B + LoRA + bf16  | ~8 GB |
| Qwen3-Emb 1.5B + LoRA + 4-bit | ~10 GB |
| Qwen3-Emb 7B + LoRA + 4-bit   | ~18 GB |